# 🔍 Binary Search Trees — Runnable Notebook

Companion to [`README.md`](README.md) and
[`03_binary_search_tree.html`](03_binary_search_tree.html).

**The one rule:** for every node, left subtree `<` node `<` right subtree — *recursively*.

## 1. Insert and search
Both follow the ordering: smaller → left, larger → right. Each step discards half the tree → `O(h)`.

In [ ]:
class TreeNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

def insert(root, val):
    """Insert by walking down as if searching; the empty slot we reach is where it belongs."""
    if root is None:
        return TreeNode(val)          # empty slot -> the new node lives here
    if val < root.val:
        root.left = insert(root.left, val)
    elif val > root.val:
        root.right = insert(root.right, val)
    # equal -> ignore duplicates (or keep a count)
    return root

def search(root, target):
    """Follow comparisons: equal=found, smaller=left, larger=right."""
    node = root
    while node:
        if target == node.val:
            return True
        node = node.left if target < node.val else node.right   # drop the other half
    return False

root = None
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    root = insert(root, v)
print("search 7 :", search(root, 7))
print("search 5 :", search(root, 5))
assert search(root, 7) and not search(root, 5)

## 2. In-order gives sorted output (the killer feature)

In [ ]:
def inorder(node, out=None):
    out = [] if out is None else out
    if node:
        inorder(node.left, out); out.append(node.val); inorder(node.right, out)
    return out

def find_min(node):
    while node.left:  node = node.left      # smallest = leftmost
    return node.val

def find_max(node):
    while node.right: node = node.right     # largest = rightmost
    return node.val

vals = inorder(root)
print("in-order (sorted!):", vals)
print("min:", find_min(root), " max:", find_max(root))
assert vals == sorted(vals)                 # the defining BST property, verified

## 3. Delete — the three cases
0 children (remove), 1 child (splice), 2 children (swap with **in-order successor**).

In [ ]:
def delete(root, val):
    """Delete `val`, keeping the BST invariant intact."""
    if root is None:
        return None
    if val < root.val:
        root.left = delete(root.left, val)
    elif val > root.val:
        root.right = delete(root.right, val)
    else:                                    # found the node to remove
        if root.left is None:                # 0 or 1 (right) child
            return root.right
        if root.right is None:               # 1 (left) child
            return root.left
        succ = root.right                    # 2 children -> in-order successor
        while succ.left:                     # = smallest value in the right subtree
            succ = succ.left
        root.val = succ.val                  # copy successor up ...
        root.right = delete(root.right, succ.val)   # ... then delete the successor
    return root

root = delete(root, 3)                       # 3 has two children (1 and 6)
print("after deleting 3:", inorder(root))
assert 3 not in inorder(root)
assert inorder(root) == sorted(inorder(root))   # STILL a valid BST

## 4. Validate a BST (the classic trap)
Checking only immediate children is **wrong** — carry a `(low, high)` allowed range down.

In [ ]:
def is_valid_bst(node, low=float("-inf"), high=float("inf")):
    """Every node must fit its allowed (low, high) window — not just beat its children."""
    if node is None:
        return True
    if not (low < node.val < high):
        return False
    # going left tightens the UPPER bound; going right tightens the LOWER bound
    return (is_valid_bst(node.left,  low, node.val) and
            is_valid_bst(node.right, node.val, high))

good = insert(insert(insert(None, 10), 5), 15)
bad = TreeNode(10)
bad.left = TreeNode(5)
bad.right = TreeNode(15)
bad.right.left = TreeNode(6)          # 6 is RIGHT of 10 but < 10  -> INVALID
print("good valid?", is_valid_bst(good))
print("bad  valid?", is_valid_bst(bad))
assert is_valid_bst(good) and not is_valid_bst(bad)

## 5. Why balance matters — count the comparisons
A **skewed** BST (sorted inserts) degenerates into a linked list: `O(n)`. A **balanced** one stays `O(log n)`.
We count search steps to see it directly.

In [ ]:
def search_steps(root, target):
    """Return how many comparisons a search makes (= the path length)."""
    node, steps = root, 0
    while node:
        steps += 1
        if target == node.val:
            break
        node = node.left if target < node.val else node.right
    return steps

# Skewed: inserting already-sorted values builds a linked list (height n-1)
skewed = None
for v in range(1, 16):
    skewed = insert(skewed, v)

# Balanced: an insert order that keeps the tree short
balanced = None
for v in [8, 4, 12, 2, 6, 10, 14, 1, 3, 5, 7, 9, 11, 13, 15]:
    balanced = insert(balanced, v)

print("searching for 15 (worst case):")
print("  skewed   tree:", search_steps(skewed, 15), "steps  -> O(n)")
print("  balanced tree:", search_steps(balanced, 15), "steps  -> O(log n)")
assert search_steps(skewed, 15) > 3 * search_steps(balanced, 15)

## ✅ Recap
- Invariant: **left `<` node `<` right**, recursively.
- Search/insert = follow comparisons, discarding half each step → `O(h)`.
- **In-order = sorted**; delete with 2 children uses the **in-order successor**.
- Validate with a **(low, high) range**, not child comparisons.
- **Skew = O(n)**; real systems use self-balancing BSTs (AVL / Red-Black).

Next: [`04_Tree_Traversal`](../04_Tree_Traversal/README.md).